## Step 1 — Import libraries and load the original data

In [1]:
# ============================================================
# INTERNOVA DATA ANALYTICS INTERNSHIP
# PROJECT: E-COMMERCE DATA ANALYSIS
# NOTEBOOK: DATA CLEANING
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# Load original dataset
input_file = Path("../dataset/messy_ecommerce_sales_data.csv")

df = pd.read_csv(input_file)

# Standardize column names
df.columns = df.columns.str.strip()

# Preserve original data
cleaned_df = df.copy()

print("Original dataset loaded successfully!")
print("Original shape:", cleaned_df.shape)

Original dataset loaded successfully!
Original shape: (103, 11)


## Step 2 — Remove exact duplicate rows

In [2]:
# Count exact duplicates
exact_duplicates = cleaned_df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

# Remove exact duplicate rows
cleaned_df = cleaned_df.drop_duplicates().copy()

print("Rows after removing exact duplicates:", len(cleaned_df))

Exact duplicate rows: 1
Rows after removing exact duplicates: 102


## Step 3 — Standardize text columns

In [3]:
# Identify text columns
text_columns = cleaned_df.select_dtypes(
    include=["object", "str", "string"]
).columns

# Remove leading and trailing spaces
for column in text_columns:
    cleaned_df[column] = cleaned_df[column].str.strip()

# Standardize product names
cleaned_df["Product"] = (
    cleaned_df["Product"].str.title()
)

# Standardize category names
category_mapping = {
    "books": "Books",
    "home": "Home",
    "sports": "Sports",
    "clothing": "Clothing",
    "electronic": "Electronics",
    "electronics": "Electronics"
}

cleaned_df["Category"] = (
    cleaned_df["Category"]
    .str.lower()
    .map(category_mapping)
)

print("Text standardization completed.")
print("\nUnique products:")
print(sorted(cleaned_df["Product"].dropna().unique()))

print("\nUnique categories:")
print(sorted(cleaned_df["Category"].dropna().unique()))

Text standardization completed.

Unique products:
['Basketball', 'Biography', 'Blender', 'Comics', 'Fiction', 'Football', 'Headphones', 'Jacket', 'Jeans', 'Lamp', 'Laptop', 'Microwave', 'Science', 'Shoes', 'Smartphone', 'Smartwatch', 'T-Shirt', 'Tennis Racket', 'Vacuum', 'Yoga Mat']

Unique categories:
['Books', 'Clothing', 'Electronics', 'Home', 'Sports']


## Step 4 — Clean Quantity

In [4]:
# Convert Quantity to numeric
cleaned_df["Quantity"] = pd.to_numeric(
    cleaned_df["Quantity"],
    errors="coerce"
)

# Replace zero and negative quantities with missing values
cleaned_df.loc[
    cleaned_df["Quantity"] <= 0,
    "Quantity"
] = np.nan

# Convert valid quantities to nullable integer
cleaned_df["Quantity"] = cleaned_df["Quantity"].astype("Int64")

print("Quantity cleaning completed.")

print(cleaned_df["Quantity"].value_counts(dropna=False))

Quantity cleaning completed.
Quantity
5       25
2       20
1       20
4       16
3       13
<NA>     8
Name: count, dtype: Int64


## Step 5 — Clean Price

In [5]:
# Convert Price to string for cleaning
cleaned_df["Price"] = (
    cleaned_df["Price"].astype("string").str.strip()
)

# Recover known text-based numeric values
price_mapping = {
    "four hundred": "400"
}

cleaned_df["Price"] = cleaned_df["Price"].replace(
    price_mapping
)

# Remove currency symbols and commas
cleaned_df["Price"] = (
    cleaned_df["Price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

# Convert to numeric
cleaned_df["Price"] = pd.to_numeric(
    cleaned_df["Price"],
    errors="coerce"
)

# Mark negative and zero prices as missing
cleaned_df.loc[
    cleaned_df["Price"] <= 0,
    "Price"
] = np.nan

print("Price cleaning completed.")

print(cleaned_df["Price"].describe())

Price cleaning completed.
count           94.0
mean      613.998191
std      1012.177998
min             38.0
25%           276.05
50%          534.705
75%         735.2625
max          10000.0
Name: Price, dtype: Float64


## Step 6 — Clean dates

In [6]:
# Convert Order_Date into datetime format
cleaned_df["Order_Date"] = pd.to_datetime(
    cleaned_df["Order_Date"],
    format="mixed",
    errors="coerce"
)

print("Date conversion completed.")

print("Missing or invalid dates:",
      cleaned_df["Order_Date"].isna().sum())

print("Date range:")
print(cleaned_df["Order_Date"].min())
print(cleaned_df["Order_Date"].max())

Date conversion completed.
Missing or invalid dates: 1
Date range:
2023-01-05 00:00:00
2025-11-06 00:00:00


## Step 7 — Investigate and fill missing categories safely

In [7]:
# Create product-category relationships
product_category_counts = (
    cleaned_df.dropna(subset=["Category"])
    .groupby("Product")["Category"]
    .nunique()
)

# Identify products with only one known category
unambiguous_products = product_category_counts[
    product_category_counts == 1
].index

# Create a mapping for unambiguous products
category_lookup = (
    cleaned_df[
        cleaned_df["Product"].isin(unambiguous_products)
        & cleaned_df["Category"].notna()
    ]
    .drop_duplicates(subset=["Product"])
    .set_index("Product")["Category"]
)

# Fill only safely identifiable missing categories
missing_before = cleaned_df["Category"].isna().sum()

cleaned_df["Category"] = cleaned_df["Category"].fillna(
    cleaned_df["Product"].map(category_lookup)
)

missing_after = cleaned_df["Category"].isna().sum()

print("Missing categories before:", missing_before)
print("Missing categories after:", missing_after)
print("Categories recovered:", missing_before - missing_after)

Missing categories before: 8
Missing categories after: 2
Categories recovered: 6


## Step 8 — Preserve Total and create gross amount

In [11]:
# Convert zero or negative quantities to missing values
cleaned_df.loc[
    cleaned_df["Quantity"] <= 0,
    "Quantity"
] = pd.NA

# Verify quantity values
print(cleaned_df["Quantity"].value_counts(dropna=False))

Quantity
5       25
2       20
1       20
4       16
3       13
<NA>     8
Name: count, dtype: Int64


In [12]:
# Calculate gross amount
cleaned_df["Gross_Amount"] = (
    cleaned_df["Quantity"] * cleaned_df["Price"]
)

# Calculate difference
cleaned_df["Total_Difference"] = (
    cleaned_df["Gross_Amount"] - cleaned_df["Total"]
)

# Classify total validation
def classify_total(row):

    if pd.isna(row["Gross_Amount"]):
        return "Insufficient Gross Data"

    if pd.isna(row["Total"]):
        return "Missing Recorded Total"

    if np.isclose(
        float(row["Gross_Amount"]),
        float(row["Total"]),
        atol=0.01
    ):
        return "Matches Gross Amount"

    return "Difference - Investigate"

cleaned_df["Total_Check"] = cleaned_df.apply(
    classify_total,
    axis=1
)

print(cleaned_df["Total_Check"].value_counts())

Total_Check
Matches Gross Amount        80
Insufficient Gross Data     15
Difference - Investigate     5
Missing Recorded Total       2
Name: count, dtype: int64


## Step 9 — Identify remaining data-quality issues

In [13]:
# Duplicate IDs after exact duplicate removal
duplicate_id_count = cleaned_df["ID"].duplicated().sum()

# Remaining missing values
missing_summary = pd.DataFrame({
    "Missing_Count": cleaned_df.isnull().sum(),
    "Missing_Percentage": (
        cleaned_df.isnull().sum() / len(cleaned_df) * 100
    ).round(2)
})

print("Remaining duplicate IDs:", duplicate_id_count)

print("\nRemaining missing values:")
display(
    missing_summary.sort_values(
        "Missing_Count",
        ascending=False
    )
)

Remaining duplicate IDs: 2

Remaining missing values:


,Missing_Count,Missing_Percentage
Total_Difference,17,16.67
Gross_Amount,15,14.71
Total,14,13.73
Original_Total,14,13.73
Quantity,8,7.84
Price,8,7.84
Category,2,1.96
Order_Date,1,0.98
ID,0,0.00
Customer_Name,0,0.00


## Step 10 — Export the cleaned dataset

In [14]:
# Save the corrected cleaned dataset
output_path = "../dataset/cleaned_ecommerce_sales.csv"

cleaned_df.to_csv(output_path, index=False)

print("Corrected cleaned dataset saved successfully!")
print("Final shape:", cleaned_df.shape)

Corrected cleaned dataset saved successfully!
Final shape: (102, 15)


In [15]:
print("FINAL DATA QUALITY REPORT")
print("=" * 45)

print("Dataset shape:", cleaned_df.shape)

print("\nMissing values:")
print(cleaned_df.isnull().sum())

print("\nDuplicate IDs:")
print(cleaned_df["ID"].duplicated().sum())

print("\nDuplicate Order IDs:")
print(cleaned_df["Order_ID"].duplicated().sum())

print("\nQuantity distribution:")
print(cleaned_df["Quantity"].value_counts(dropna=False))

print("\nTotal validation:")
print(cleaned_df["Total_Check"].value_counts())

print("\nNegative quantities remaining:")
print((cleaned_df["Quantity"] < 0).sum())

print("\nNegative prices remaining:")
print((cleaned_df["Price"] < 0).sum())

FINAL DATA QUALITY REPORT
Dataset shape: (102, 15)

Missing values:
ID                   0
Customer_Name        0
Order_ID             0
Order_Date           1
Product              0
Category             2
Quantity             8
Price                8
Payment_Method       0
Status               0
Total               14
Original_Total      14
Gross_Amount        15
Total_Difference    17
Total_Check          0
dtype: int64

Duplicate IDs:
2

Duplicate Order IDs:
2

Quantity distribution:
Quantity
5       25
2       20
1       20
4       16
3       13
<NA>     8
Name: count, dtype: Int64

Total validation:
Total_Check
Matches Gross Amount        80
Insufficient Gross Data     15
Difference - Investigate     5
Missing Recorded Total       2
Name: count, dtype: int64

Negative quantities remaining:
0

Negative prices remaining:
0
